# E-Commerce Dress Sales Analytics\n## Exploratory Data Analysis (EDA)\n\nGoogle Colab / Python 3 compatible.

In [ ]:
from google.colab import files\nuploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

In [ ]:
FILE_NAME = "dress_sales.csv"\ndf = pd.read_csv(FILE_NAME)\nprint("Shape:", df.shape)\ndisplay(df.head())

## 1. Data Understanding

In [ ]:
display(df.dtypes.to_frame("Data Type"))\ndisplay(df.describe(include="all").T)

## 2. Data Cleaning

In [ ]:
print("Missing values:")\ndisplay(df.isnull().sum().to_frame("Missing Count"))\nprint("Duplicate rows:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates().copy()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    if not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining missing values:", df.isnull().sum().sum())

## 3. Feature Engineering

In [ ]:
df["Gross_Sales"] = df["Price"] * df["Quantity"]
df["Discount_Amount"] = df["Gross_Sales"] * (df["Discount"] / 100)
df["Net_Sales"] = df["Gross_Sales"] - df["Discount_Amount"]
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.month_name()
df["Day_of_Week"] = df["Date"].dt.day_name()
display(df.head())

## 4. Business KPIs

In [ ]:
total_revenue = df["Net_Sales"].sum()
total_orders = df["Order_ID"].nunique()
total_items = df["Quantity"].sum()
aov = total_revenue / total_orders if total_orders else 0

print(f"Total Revenue: {total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Total Items Sold: {total_items:,}")
print(f"Average Order Value: {aov:,.2f}")

## 5. Category Analysis

In [ ]:
category_summary = df.groupby("Category").agg(
    Revenue=("Net_Sales", "sum"),
    Quantity=("Quantity", "sum"),
    Orders=("Order_ID", "nunique"),
    Average_Rating=("Rating", "mean")
).sort_values("Revenue", ascending=False)

display(category_summary)

In [ ]:
px.bar(
    category_summary.reset_index(),
    x="Category", y="Revenue",
    title="Revenue by Dress Category",
    text_auto=".2s"
).show()

## 6. Top-Selling Products

In [ ]:
product_summary = df.groupby("Product_ID").agg(
    Quantity_Sold=("Quantity", "sum"),
    Revenue=("Net_Sales", "sum")
).sort_values("Quantity_Sold", ascending=False)

display(product_summary.head(10))

In [ ]:
px.bar(
    product_summary.head(10).reset_index(),
    x="Product_ID", y="Quantity_Sold",
    title="Top 10 Products by Quantity Sold",
    text_auto=True
).show()

## 7. Monthly Sales Trend

In [ ]:
monthly = df.groupby(["Year", "Month"]).agg(
    Revenue=("Net_Sales", "sum"),
    Quantity=("Quantity", "sum")
).reset_index()
monthly["Period"] = monthly["Year"].astype(str) + "-" + monthly["Month"].astype(str).str.zfill(2)
display(monthly)

In [ ]:
px.line(
    monthly, x="Period", y="Revenue",
    markers=True, title="Monthly Revenue Trend"
).show()

## 8. Discount and Rating Analysis

In [ ]:
px.scatter(
    df, x="Discount", y="Quantity",
    color="Category",
    hover_data=["Price", "Rating"],
    title="Discount vs Quantity Sold"
).show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="Rating", y="Quantity")
plt.title("Rating vs Quantity Sold")
plt.show()

## 9. Customer Analysis

In [ ]:
customer_summary = df.groupby("Customer_ID").agg(
    Total_Spending=("Net_Sales", "sum"),
    Total_Items=("Quantity", "sum"),
    Total_Orders=("Order_ID", "nunique")
).sort_values("Total_Spending", ascending=False)

display(customer_summary.head(10))

## 10. Correlation Analysis

In [ ]:
plt.figure(figsize=(10,7))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

## EDA Conclusion\nUse the actual outputs to identify the strongest business findings. Do not assume a category, month, or feature is the best until the dataset results confirm it.